# Real Estate Crawling: Filtering, Concurrency & Sitemaps

This notebook demonstrates three features for turning a broad link-follower
into a focused **document-inventory** crawler:

1. **Configurable concurrency** — tune throughput with `max_concurrency` and
   the connection-pool limits.
2. **URL filtering rules** (`CrawlRules`) — follow only the URLs you care
   about and skip crawl traps (login pages, sort/calendar links, binaries,
   off-site domains).
3. **Sitemap discovery** — seed the queue from a site's own published
   document inventory before link-following begins.

We use a municipal property/GIS site whose street pages link to many
`?Letter=` query-parameter variants — a classic crawl trap that filtering
rules are designed to tame.

In [ ]:
import logging
import time
from urllib.parse import urlparse, parse_qs

from linktrace import Spider, CrawlRules, Crawler, SitemapParser

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

START_URL = "https://gis.vgsi.com/WoonsocketRI/Streets.aspx"
CACHE = ".feature_demo_cache"  # cache so repeated runs are fast

## 1. Configurable concurrency

`max_concurrency` controls how many URLs are fetched per batch (default 10),
while `max_connections` / `max_connections_per_host` size the underlying
aiohttp connection pool. Raise them to go faster across many hosts; keep
`max_connections_per_host` modest to stay polite to a single server.

In [ ]:
spider = Spider(
    start_url=START_URL,
    max_depth=1,
    max_concurrency=20,            # up to 20 fetches per batch (default 10)
    max_connections=100,           # total pool size
    max_connections_per_host=6,    # be gentle on this one host
    cache_dir=CACHE,
    show_progress=True,
)

start = time.time()
documents = await spider.run_async()
elapsed = time.time() - start

print(f"\nCrawled {len(documents)} pages in {elapsed:.1f}s "
      f"(max_concurrency={spider.max_concurrency})")

## 2. URL filtering rules — taming query-param crawl traps

Each street page links to dozens of `?Letter=A`, `?Letter=B`, ... variants.
Without rules those all get queued. Let's first measure how many discovered
links carry a `Letter` parameter.

In [ ]:
discovered = [link.url for doc in documents for link in doc.internal_links]
with_param = [u for u in discovered if 'Letter=' in u]

print(f"Total internal links discovered: {len(discovered)}")
print(f"  ...carrying a Letter= query param: {len(with_param)}")
print("\nExamples:")
for u in with_param[:5]:
    print(f"  {u}")

Now define `CrawlRules` to keep the crawl on-site and drop the parameter
variants, binaries, and any off-domain links. We compare how many links
survive filtering.

In [ ]:
rules = CrawlRules(
    allowed_domains=["gis.vgsi.com"],   # stay on-site (subdomain-aware)
    exclude_query_params=["Letter"],     # drop the ?Letter= crawl trap
    blocked_extensions=["pdf", "jpg", "png", "zip"],
)

kept = [u for u in discovered if rules.allows(u)]
dropped = [u for u in discovered if not rules.allows(u)]

print(f"Links kept by rules:    {len(kept)}")
print(f"Links dropped by rules: {len(dropped)}")
print("\nSample kept:")
for u in kept[:5]:
    print(f"  {u}")

### How rules are evaluated

Rules are checked against every **discovered** link before it is queued (the
`start_url` is always fetched). For each dimension a populated *allow* list
is a whitelist and a *block*/*exclude* list rejects matches — and
**exclusions always win**. The `allows()` method is pure, so it's easy to
unit-test your policy in isolation:

In [ ]:
preset = CrawlRules(
    allowed_domains=["realty.example.com"],
    include_path_prefixes=["/homes-for-sale/", "/agents/"],
    exclude_path_prefixes=["/login", "/privacy"],
    blocked_extensions=["pdf", "jpg"],
    exclude_query_params=["sort", "view", "calendar"],
    exclude_patterns=[r"/print/?$"],
)

checks = [
    "https://realty.example.com/homes-for-sale/123",        # keep
    "https://realty.example.com/agents/jane",               # keep
    "https://realty.example.com/login",                     # exclude prefix
    "https://realty.example.com/homes-for-sale/1?sort=asc", # exclude param
    "https://realty.example.com/homes-for-sale/flyer.pdf",  # exclude ext
    "https://realty.example.com/homes-for-sale/1/print",    # exclude regex
    "https://other-site.com/homes-for-sale/1",              # off-domain
]
for url in checks:
    verdict = "KEEP " if preset.allows(url) else "DROP "
    print(f"  {verdict} {url}")

## 3. Sitemap discovery

Link-following only finds pages reachable from the start URL. Sitemaps expose
a site's full document inventory. `SitemapParser.parse()` is namespace-
agnostic and handles both `<urlset>` listings and `<sitemapindex>` files.

In [ ]:
URLSET = b'''<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
  <url><loc>https://realty.example.com/homes-for-sale/1</loc></url>
  <url><loc>https://realty.example.com/homes-for-sale/2</loc></url>
</urlset>'''

kind, locs = SitemapParser.parse(URLSET)
print(f"Parsed a {kind} with {len(locs)} URLs:")
for u in locs:
    print(f"  {u}")

Against a live site, `Crawler.discover_sitemap_urls()` reads `Sitemap:`
declarations from `robots.txt` (falling back to `/sitemap.xml`), follows
nested sitemap indexes, and returns the page URLs. It's best-effort — if no
sitemap exists it simply returns an empty list.

In [ ]:
async with Crawler(cache_dir=CACHE) as crawler:
    sitemap_urls = await crawler.discover_sitemap_urls(START_URL)

print(f"Sitemap discovery returned {len(sitemap_urls)} URL(s)")
for u in sitemap_urls[:10]:
    print(f"  {u}")
if not sitemap_urls:
    print("  (this site publishes no sitemap — link-following still works)")

## 4. Putting it together — a focused real-estate preset

Combine all three: seed from sitemaps, filter discovered links, and tune
concurrency. The same `CrawlRules` is applied to sitemap-seeded URLs *and*
links found while crawling.

In [ ]:
focused = Spider(
    start_url=START_URL,
    max_depth=2,
    use_sitemaps=True,                  # seed from sitemap.xml / robots.txt
    rules=CrawlRules(                   # ...and filter everything
        allowed_domains=["gis.vgsi.com"],
        exclude_query_params=["Letter"],
        blocked_extensions=["pdf", "jpg", "png"],
    ),
    max_concurrency=20,
    cache_dir=CACHE,
    show_progress=True,
)

docs = await focused.run_async()
print(f"\nFocused crawl fetched {len(docs)} pages")

# Confirm the Letter= trap was avoided
trap = [d for d in docs if 'Letter=' in d.url]
print(f"Pages with a Letter= param (should be 0): {len(trap)}")

## Takeaways

- **`max_concurrency`** + connection-pool limits tune throughput without
  touching the rest of your config.
- **`CrawlRules`** turns an exhaustive link-follower into a targeted
  document crawler; `allows()` is pure and unit-testable.
- **`use_sitemaps=True`** discovers pages link-following would miss, and the
  same rules keep that inventory on-topic.

See the [Examples](https://linktrace.readthedocs.io/examples/) and
[API Reference](https://linktrace.readthedocs.io/api-reference/) docs for the
full parameter list.